# T2M Diagnostic Benchmark — Evaluation Runner

This notebook evaluates **pre-generated, standardised Text-to-Motion `.npy` files**. The code lives in the team GitHub repository; this notebook only loads data, calls the code and shows results.

| Path | Contents | Owner |
|---|---|---|
| `evaluation/common.py` | settings, input contract, loading, evaluation cases, `BaseEvaluator` / registry, Human Gold helpers | whole team (PR + review by all) |
| `evaluation/eval_trajectory.py` | `TrajectoryEvaluator`, direction ratio, `DirectionDecisionRule` | member A |
| `evaluation/eval_body_side.py` | `BodySideEvaluator` + calibration helpers | member B |
| `evaluation/eval_<name>.py` | further evaluators (copy `eval_template.py`) | members C, D |
| `generation/<model>/` | model runners that produce the standardised motion ZIP | one member per model |
| `benchmark/` | benchmark definition JSON (single source of truth) | whole team |
| `labels/` | Human Gold Label JSON per model | annotators |

## How to Use This Notebook — Multi-model Pilot Evaluation

This notebook is designed to evaluate multiple Text-to-Motion models using the **same benchmark and evaluator rules**.

### Common workflow for each model

`Standardised .npy`  
→ Load Benchmark Prompts / Requirements  
→ Create or Load Human Gold Labels  
→ Extract Requirement-level Evidence  
→ Apply Automatic Evaluation Rules  
→ Compare with Human Gold

### During the Pilot Stage

During the pilot stage, evaluator rules and thresholds can be developed and validated. Mismatches with Human Gold should be analysed to determine whether a rule needs refinement.

### Final Benchmark

After Cross-model Pilot Validation is completed, the evaluator rules should be **frozen**.

The same frozen rules must then be applied to every model in the final benchmark so that the models are compared under the same evaluation criteria.

# 【Setup】

## STEP 0 — Get the code from GitHub

Clones the repository (first run) or pulls the latest version, then imports the shared code and all evaluators.

- `BRANCH = "main"` for normal use. To test your own evaluator before it is merged, set `BRANCH` to your branch name.
- Private repository: add a GitHub token as a Colab secret named `GITHUB_TOKEN` (key icon in the left sidebar).

In [ ]:
import importlib, os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/<team>/t2m-benchmark.git"   # ← change once
BRANCH = "main"                                            # or your feature branch
REPO_DIR = Path("/content/t2m-benchmark")

def _git(*args, cwd=None):
    print("$ git", " ".join(args))
    subprocess.run(["git", *args], cwd=cwd, check=True)

url = REPO_URL
try:  # optional token for a private repository
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if token:
        url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass

if not (REPO_DIR / ".git").exists():
    _git("clone", "--branch", BRANCH, url, str(REPO_DIR))
else:
    _git("fetch", "origin", cwd=REPO_DIR)
    _git("checkout", BRANCH, cwd=REPO_DIR)
    _git("pull", "origin", BRANCH, cwd=REPO_DIR)

EVAL_DIR = REPO_DIR / "evaluation"          # common.py and eval_*.py live here
if str(EVAL_DIR) not in sys.path:
    sys.path.insert(0, str(EVAL_DIR))

# Import (or re-import after a pull) the shared code and every evaluator.
import common, eval_trajectory, eval_body_side
for module in (common, eval_trajectory, eval_body_side):
    importlib.reload(module)

from common import *                       # settings, loaders, registry, gold helpers
from eval_trajectory import TrajectoryEvaluator, calculate_required_direction_ratio, DirectionDecisionRule
from eval_body_side import BodySideEvaluator

commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR,
                        capture_output=True, text=True).stdout.strip()
print("\nFramework commit     :", commit or "unknown")
print("Registered evaluators:", sorted(EVALUATOR_REGISTRY))

# 【Phase A — Benchmark Input Preparation】

## STEP 3 — Benchmark Input Contract

All input motions must satisfy the same Standardised Motion Contract before evaluation.

Expected format:

- Shape: `[T, 22, 3]`
- Global XYZ coordinates
- `+Y` = Up
- `+Z` = Forward
- `+X` = Right
- Unit = metres
- Frame rate ≈ 20 fps

The benchmark evaluates the contents of the `.npy` motion file, not the file format itself. Model-specific outputs must therefore be converted to this common representation before use.

## STEP 4 — Select the Evaluation Model

Each member changes only `MODEL_NAME`. The motion package must contain `<MODEL_NAME>/<prompt_id>.npy`.

> Model-specific generation and coordinate conversion are done in `generation/<model>/`, outside this notebook.

In [ ]:
# ============================================================
# STEP 4 — Select Model and Standardised Motion Folder
# ============================================================

MODEL_NAME = "MoMADiff"       # ← each member changes only this value

BENCHMARK_INPUT_ROOT = Path("/content/benchmark_inputs")
MODEL_INPUT_DIR = BENCHMARK_INPUT_ROOT / MODEL_NAME

print("=" * 80)
print("MODEL SELECTION")
print("=" * 80)
print("Model       :", MODEL_NAME)
print("Input Folder:", MODEL_INPUT_DIR)

## STEP 3.5 — Import the Pre-generated Motion Package

Upload `<MODEL_NAME>_standardized_pilot.zip`. Skipped if the model folder already exists.

In [ ]:
# ============================================================
# STEP 3.5 — Import Pre-generated Motion Package
# ============================================================

if MODEL_INPUT_DIR.exists():
    print("Motion folder already present — upload skipped:", MODEL_INPUT_DIR)
else:
    from google.colab import files
    print("Upload the standardized motion ZIP for", MODEL_NAME)
    uploaded = files.upload()
    zip_files = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_files) != 1:
        raise RuntimeError(f"Expected exactly 1 ZIP file, found {len(zip_files)}.")
    extract_motion_package(zip_files[0], BENCHMARK_INPUT_ROOT)

if not MODEL_INPUT_DIR.exists():
    raise FileNotFoundError(
        f"Model input folder not found: {MODEL_INPUT_DIR}\n"
        "Check MODEL_NAME and the uploaded motion package."
    )
print("\nSTEP 3.5 — Motion package ready ✅")

## STEP 5 — Load the Benchmark Definition

Read from the repository (`benchmark/`), so every member uses the same version. If the file is not in the repository yet, you are asked to upload it.

In [ ]:
# ============================================================
# STEP 5 — Load Pilot Benchmark Definition
# ============================================================

BENCHMARK_FILE = REPO_DIR / "benchmark" / "pilot_benchmark_definition.json"   # ← file name in the repo

if BENCHMARK_FILE.exists():
    benchmark_definition, pilot_prompts = load_benchmark_definition(BENCHMARK_FILE)
    benchmark_source = str(BENCHMARK_FILE.relative_to(REPO_DIR))
else:
    from google.colab import files
    print(f"{BENCHMARK_FILE.name} is not in the repository — please upload the Pilot Benchmark JSON.")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file was uploaded.")
    benchmark_source = next(iter(uploaded))
    benchmark_definition, pilot_prompts = parse_benchmark_definition(uploaded[benchmark_source], benchmark_source)

print_benchmark_summary(benchmark_definition, pilot_prompts, benchmark_source)
print("STEP 5 — Pilot Benchmark loaded ✅")

## STEP 6 — Load and Validate Pre-generated `.npy` Motions

Load each Pilot motion from the selected model folder and verify that it satisfies the benchmark input requirements.

The expected file naming convention is based on the Prompt ID, for example:

`C1-01.npy`

`C1-02.npy`

The motion files must already be standardised before this step.

In [ ]:
# ============================================================
# STEP 6 — Load and Validate Pre-generated Motions
# ============================================================

registered_motions, input_results = register_motions(pilot_prompts, MODEL_NAME, MODEL_INPUT_DIR)

# 【Phase B — Automatic Evaluation】

## STEP 7 — Evaluation Case Construction

Construct the evaluation cases by linking each Pilot Prompt and its Atomic Requirements to the corresponding standardised motion file.

Evaluation is performed at the **requirement level**, rather than only assigning one overall score to the complete prompt.

In [ ]:
# ============================================================
# STEP 7 — Evaluation Case Construction
# ============================================================

evaluation_cases = build_evaluation_cases(pilot_prompts, registered_motions, MODEL_NAME)

## STEP 8 — Evaluation Configuration

`EVALUATION_CONFIG` (in `common.py`) maps each Atomic Requirement type to its evaluator. The table also shows which evaluators are already implemented and registered.

In [ ]:
used_requirement_types, missing_types, evaluator_status = check_evaluation_config(evaluation_cases)

# STEP 9C — Human Gold Label Template

Create a template for Human Gold Label entry from the Evaluation Cases.

Human Gold Labels are assigned by observing the **motion generated by each model**. Therefore, even for the same prompt, Human Gold may differ between models.

For each Atomic Requirement, the template allows one of the following labels:

- `PASS`
- `FAIL`
- `UNCERTAIN`

In [ ]:
# ============================================================
# STEP 9C — Human Gold Label Template
# ============================================================

human_gold_template = make_gold_template(evaluation_cases)
GOLD_FILE = REPO_DIR / "labels" / gold_label_filename(MODEL_NAME)
print("\nHuman Gold file for this model:", GOLD_FILE.relative_to(REPO_DIR),
      "(exists)" if GOLD_FILE.exists() else "(not created yet)")

# STEP 9D — Human Gold Label Entry (only when creating new labels)

Enter `P` / `F` / `U` for each Atomic Requirement. Skip this and STEP 9E if `labels/<MODEL_NAME>_pilot_human_gold_labels.json` already exists in the repository — go to STEP 9F.

In [ ]:
# ============================================================
# STEP 9D — Human Gold Label Entry
# ============================================================

HUMAN_GOLD_LABELS = enter_gold_labels(human_gold_template, MODEL_NAME)

# STEP 9E — Save Human Gold Labels (only when creating new labels)

Saves the labels into the local clone (`labels/`) and downloads a copy. **To share them, add the file to `labels/` in GitHub through a Pull Request** (GitHub → Add file → Upload files → create a new branch).

In [ ]:
# ============================================================
# STEP 9E — Save Human Gold Labels
# ============================================================

saved = save_gold_labels(HUMAN_GOLD_LABELS, GOLD_FILE)
print("Saved:", saved)
try:
    from google.colab import files
    files.download(str(saved))
except Exception:
    pass

# STEP 9F — Load Saved Human Gold Labels

Loads `labels/<MODEL_NAME>_pilot_human_gold_labels.json` from the repository. If it is not there, you are asked to upload it.

**Human Gold is model-specific** — each model generates different motions.

In [ ]:
# ============================================================
# STEP 9F — Load Saved Human Gold Labels
# ============================================================

if GOLD_FILE.exists():
    gold_source = GOLD_FILE
else:
    from google.colab import files
    print(f"{GOLD_FILE.name} not found in the repository — upload the Human Gold JSON for {MODEL_NAME}.")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No Human Gold JSON was uploaded.")
    gold_source = Path(next(iter(uploaded)))

HUMAN_GOLD_LABELS, label_counts = load_gold_labels(gold_source)

print("\n" + "=" * 80)
print("HUMAN GOLD LABELS LOADED")
print("=" * 80)
print("Model              :", MODEL_NAME)
print("Source             :", gold_source)
print("Pilot Prompts      :", len(HUMAN_GOLD_LABELS))
print("Total Requirements :", label_counts["total"])
print("PASS               :", label_counts["PASS"])
print("FAIL               :", label_counts["FAIL"])
print("UNCERTAIN          :", label_counts["UNCERTAIN"])
print("=" * 80)

## STEP 10 — Direction Requirements (`eval_trajectory.py`, owner: member A)

The evaluator code is in `evaluation/eval_trajectory.py`; the cells below run the same calibration workflow as Df5 (10A–10F).

## STEP 10A — TrajectoryEvaluator

This step defines the evaluator used to collect **Trajectory Evidence** for Direction Requirements (`forward`, `backward`, `left`, and `right`).

### What this evaluator measures

The evaluator extracts the trajectory of the root joint (Joint 0) from the Standardised Motion `[T, 22, 3]` and analyses movement on the XZ plane.

The benchmark coordinate system is:

- `+X` = Right
- `-X` = Left
- `+Z` = Forward
- `-Z` = Backward
- `+Y` = Up

For each motion, the evaluator calculates:

- `dx` — displacement along the X-axis
- `dz` — displacement along the Z-axis
- `total_displacement` — straight-line displacement on the XZ plane
- `dominant_axis` — whether X or Z has the larger displacement
- `dominance_ratio` — relative dominance of the larger axis
- `raw_direction` — preliminary direction estimated from the displacement

### Important

This step **does not yet make the final PASS / FAIL decision**. The evaluator first collects measurable Evidence. The final Direction rule and thresholds are calibrated later using the Pilot Human Gold Labels.

**Input:** Standardised Motion `[T, 22, 3]`  
**Output:** Direction Trajectory Evidence

### STEP 10A-1 — Quick Sanity Check

Before running calibration on all Pilot cases, test the `TrajectoryEvaluator` on one registered motion with a `direction = left` Requirement.

This verifies that:

- the `.npy` motion can be loaded correctly,
- the root trajectory can be extracted,
- `dx`, `dz`, dominant axis, dominance ratio, and raw direction are produced,
- PASS / FAIL remains unset before threshold calibration.

This cell is only a **sanity check** and does not determine the final Direction rule.

In [ ]:
# Find an Evaluation Case containing a direction=left Requirement
left_case = None
for case in evaluation_cases:
    for req in case["requirements"]:
        value = req.get("value", req.get("expected"))
        if req.get("type") == "direction" and str(value).lower() == "left":
            left_case = case
            break
    if left_case is not None:
        break

if left_case is None:
    raise RuntimeError("No Evaluation Case with direction=left was found.")

motion = np.load(left_case["motion_path"], allow_pickle=False)
motion = validate_standardised_motion(motion)

evaluator = TrajectoryEvaluator()
result = evaluator.evaluate(motion=motion, required_direction="left")
evidence = result["evidence"]

print("Motion used:", left_case["motion_path"])
print("Shape      :", motion.shape)
print("Required Direction:", result["required_direction"])
print("Predicted Direction:", result["predicted_direction_raw"])
print(f"dx: {evidence['dx']:.4f} m")
print(f"dz: {evidence['dz']:.4f} m")
print("Dominant axis :", evidence["dominant_axis"])
print("Dominance ratio:", evidence["dominance_ratio"])
print("PASS / FAIL   :", result["pass_fail"], "(Threshold not set)")

## STEP 10B — Collect Direction Evidence and Link Human Gold

Apply `TrajectoryEvaluator` to every Pilot case containing a `direction` Requirement and link each Requirement to the Human Gold Label created in STEP 9D.

For each Direction Requirement, this step records:

- required direction,
- Human Gold Label,
- raw predicted direction,
- X and Z displacement,
- displacement in the required direction,
- orthogonal displacement,
- required-direction ratio.

The purpose is to create the calibration dataset used to design the Direction decision rule.

`UNCERTAIN` labels are retained here for inspection, but they should not be treated as PASS or FAIL when fitting a threshold.

In [ ]:
# get_human_label / calculate_required_direction_ratio come from common.py / eval_trajectory.py

# ============================================================
# 3. Direction Requirementを持つEvaluation Caseを評価
# ============================================================

direction_results = []

evaluator = TrajectoryEvaluator()


for case in evaluation_cases:

    # --------------------------------------------------------
    # RequirementをIndex付きで取得
    # --------------------------------------------------------

    for req_index, req in enumerate(
        case["requirements"]
    ):

        # Direction Requirement以外は今回は無視
        if req.get("type") != "direction":
            continue


        # ----------------------------------------------------
        # Required Direction
        # ----------------------------------------------------

        required_direction = req.get(
            "value",
            req.get("expected")
        )

        if required_direction is None:
            continue

        required_direction = str(
            required_direction
        ).lower()


        # ----------------------------------------------------
        # Prompt ID
        # ----------------------------------------------------

        prompt_id = (
            case.get("prompt_id")
            or case.get("id")
            or case.get("case_id")
            or "UNKNOWN"
        )


        # ----------------------------------------------------
        # Motion読み込み
        # ----------------------------------------------------

        motion = np.load(
            case["motion_path"],
            allow_pickle=False
        )

        motion = validate_standardised_motion(
            motion
        )


        # ----------------------------------------------------
        # TrajectoryEvaluator
        # ----------------------------------------------------

        result = evaluator.evaluate(
            motion=motion,
            required_direction=required_direction
        )

        evidence = result["evidence"]

        dx = evidence["dx"]
        dz = evidence["dz"]


        # ----------------------------------------------------
        # Required-direction Evidence
        # ----------------------------------------------------

        (
            required_displacement,
            orthogonal_displacement,
            required_ratio,
        ) = calculate_required_direction_ratio(

            dx=dx,

            dz=dz,

            required_direction=required_direction,
        )


        # ----------------------------------------------------
        # Human Gold Label
        #
        # STEP 9DのHUMAN_GOLD_LABELSから取得
        # ----------------------------------------------------

        human_label = get_human_label(
            HUMAN_GOLD_LABELS,
            case=case,
            requirement_index=req_index,
        )


        # ----------------------------------------------------
        # 結果を保存
        # ----------------------------------------------------

        direction_results.append({

            "prompt_id":
                prompt_id,

            "requirement_index":
                req_index,

            "motion_path":
                case["motion_path"],

            "required_direction":
                required_direction,

            "human_label":
                human_label,

            "predicted_raw":
                result["predicted_direction_raw"],

            "dx":
                dx,

            "dz":
                dz,

            "required_displacement":
                required_displacement,

            "orthogonal_displacement":
                orthogonal_displacement,

            "required_ratio":
                required_ratio,

            "dominant_axis":
                evidence["dominant_axis"],

            "dominance_ratio":
                evidence["dominance_ratio"],
        })


# ============================================================
# 4. Display results
# ============================================================

print(
    f"Direction Requirement Cases: "
    f"{len(direction_results)}"
)

print("=" * 100)


for i, r in enumerate(
    direction_results,
    start=1
):

    print(f"\n[{i}]")

    print(
        "Prompt ID     :",
        r["prompt_id"]
    )

    print(
        "Req Index     :",
        r["requirement_index"]
    )

    print(
        "Motion        :",
        r["motion_path"]
    )

    print(
        "Required      :",
        r["required_direction"]
    )

    print(
        "Human Label   :",
        r["human_label"]
    )

    print(
        "Predicted Raw :",
        r["predicted_raw"]
    )

    print(
        f"dx            : "
        f"{r['dx']:.4f} m"
    )

    print(
        f"dz            : "
        f"{r['dz']:.4f} m"
    )

    print(
        f"Required Disp : "
        f"{r['required_displacement']:.4f} m"
    )

    print(
        f"Orthogonal    : "
        f"{r['orthogonal_displacement']:.4f} m"
    )

    print(
        f"Required Ratio: "
        f"{r['required_ratio']:.4f}"
    )

    print("-" * 100)


# ============================================================
# 5. Human Label確認
# ============================================================

print("\n" + "=" * 100)
print("HUMAN LABEL SUMMARY")
print("=" * 100)

pass_count = sum(
    r["human_label"] == "PASS"
    for r in direction_results
)

fail_count = sum(
    r["human_label"] == "FAIL"
    for r in direction_results
)

uncertain_count = sum(
    r["human_label"] == "UNCERTAIN"
    for r in direction_results
)

none_count = sum(
    r["human_label"] is None
    for r in direction_results
)


print("PASS      :", pass_count)
print("FAIL      :", fail_count)
print("UNCERTAIN :", uncertain_count)
print("None      :", none_count)


if none_count == 0:
    print("\nHuman Gold Labels linked successfully ✅")
else:
    print(
        "\nWARNING: Human Labelが取得できていない"
        "Requirementがあります。"
    )


print("=" * 100)
print("STEP 10B — Direction Evidence collected ✅")
print("=" * 100)

## STEP 10C — Candidate Minimum-Displacement Threshold Search

Use the Pilot Direction cases from STEP 10B to compare candidate **minimum required-displacement thresholds**.

For calibration:

- Human `PASS` and `FAIL` cases are used.
- Human `UNCERTAIN` cases are excluded from threshold fitting.
- Each candidate threshold produces an automatic PASS / FAIL prediction.
- The prediction is compared with Human Gold using Accuracy, False PASS, and False FAIL.

### Important

The best value found here is a **Pilot candidate threshold**, not yet the final benchmark threshold.

The Direction rule can be refined using Pilot evidence and cross-model validation. Once the evaluator design is finalised, the rule should be **frozen before evaluation of the remaining benchmark prompts**.

In [ ]:
# ============================================================
# STEP 10C — Candidate Direction Threshold Search
# Compare Minimum Displacement candidates using Pilot Human Gold
# ============================================================

import numpy as np


# ------------------------------------------------------------
# 1. Candidate Thresholds used for calibration
# ------------------------------------------------------------

candidate_thresholds = [
    0.10,
    0.25,
    0.50,
    0.75,
    1.00,
    1.25,
]


# ------------------------------------------------------------
# 2. Exclude UNCERTAIN
# ------------------------------------------------------------

calibration_cases = [
    r for r in direction_results
    if r["human_label"] in {"PASS", "FAIL"}
]


print("=" * 100)
print("DIRECTION THRESHOLD CALIBRATION")
print("=" * 100)

print(
    "Usable Human-labelled cases:",
    len(calibration_cases)
)

print(
    "UNCERTAIN excluded:",
    len(direction_results) - len(calibration_cases)
)


# ------------------------------------------------------------
# 3. Evaluate each Candidate Threshold
# ------------------------------------------------------------

threshold_results = []


for threshold in candidate_thresholds:

    correct = 0
    total = 0

    false_pass = 0
    false_fail = 0

    case_results = []


    for r in calibration_cases:

        required_disp = r[
            "required_displacement"
        ]

        human_label = r[
            "human_label"
        ]


        # ----------------------------------------------------
        # Candidate Direction Rule
        #
        # 1. Movement in the required direction is correct
        # 2. Required displacement >= threshold
        # ----------------------------------------------------

        if required_disp >= threshold:

            predicted_label = "PASS"

        else:

            predicted_label = "FAIL"


        # ----------------------------------------------------
        # Compare with Human Gold
        # ----------------------------------------------------

        is_correct = (
            predicted_label == human_label
        )

        if is_correct:
            correct += 1

        elif (
            predicted_label == "PASS"
            and human_label == "FAIL"
        ):
            false_pass += 1

        elif (
            predicted_label == "FAIL"
            and human_label == "PASS"
        ):
            false_fail += 1


        total += 1


        case_results.append({

            "prompt_id":
                r["prompt_id"],

            "human_label":
                human_label,

            "predicted_label":
                predicted_label,

            "required_displacement":
                required_disp,

            "correct":
                is_correct,
        })


    # --------------------------------------------------------
    # Accuracy
    # --------------------------------------------------------

    accuracy = (
        correct / total
        if total > 0
        else 0.0
    )


    threshold_results.append({

        "threshold":
            threshold,

        "correct":
            correct,

        "total":
            total,

        "accuracy":
            accuracy,

        "false_pass":
            false_pass,

        "false_fail":
            false_fail,

        "cases":
            case_results,
    })


# ------------------------------------------------------------
# 4. Compare thresholds
# ------------------------------------------------------------

print("\n")
print(
    f"{'Threshold':<12}"
    f"{'Correct':<12}"
    f"{'Accuracy':<12}"
    f"{'False PASS':<14}"
    f"{'False FAIL':<14}"
)

print("-" * 70)


for result in threshold_results:

    print(
        f"{result['threshold']:<12.2f}"
        f"{result['correct']:<12}"
        f"{result['accuracy']:<12.3f}"
        f"{result['false_pass']:<14}"
        f"{result['false_fail']:<14}"
    )


# ------------------------------------------------------------
# 5. Check best accuracy
# ------------------------------------------------------------

best_accuracy = max(
    r["accuracy"]
    for r in threshold_results
)

best_candidates = [
    r for r in threshold_results
    if r["accuracy"] == best_accuracy
]


print("\n" + "=" * 100)
print("BEST CANDIDATE(S)")
print("=" * 100)


for result in best_candidates:

    print(
        f"Threshold = "
        f"{result['threshold']:.2f} m"
        f" | Accuracy = "
        f"{result['accuracy']:.3f}"
    )


print("=" * 100)
print(
    "NOTE: These are PILOT candidate thresholds, "
    "not final benchmark thresholds."
)
print("=" * 100)

## STEP 10D — Define the Initial Direction Rule

Based on the Pilot calibration results from STEP 10C, define the **Initial Direction Decision Rule**.

### Initial Pilot Rule

A Direction requirement is predicted as PASS only when both conditions are satisfied:

1. The motion has the correct sign for the required direction.
2. The displacement in the required direction is at least the Minimum Displacement Threshold.

Direction sign conditions:

- `forward` → `dz > 0`
- `backward` → `dz < 0`
- `right` → `dx > 0`
- `left` → `dx < 0`

Because a Human-PASS Pilot motion can move diagonally, `dominant_axis` is **not** used as a mandatory PASS condition. `dominant_axis` and `dominance_ratio` are retained as diagnostic evidence.

### Current Status

**Pilot Candidate — NOT FROZEN**

Current candidate threshold: **0.50 m**

The same initial rule must first be applied to all Pilot models. **Do not tune the threshold separately for each model.**

After Cross-model Validation, the rule and/or threshold can be refined if the evidence shows that a change is necessary.

`DirectionDecisionRule` is defined in `eval_trajectory.py` (imported in STEP 0).

## STEP 10E — Validate the Initial Direction Rule on the Current Model Pilot

Apply the Initial Direction Rule from STEP 10D to the Pilot Direction cases for the model selected in STEP 4.

Compare the Automatic Prediction with that model's Human Gold Labels and inspect:

- Accuracy
- False PASS
- False FAIL
- Mismatch cases

Human Gold cases labelled `UNCERTAIN` are excluded from the accuracy calculation.

This is **Current Model Pilot Validation**, not the final rule confirmation.

In [ ]:
# ============================================================
# STEP 10E — Current Model Pilot Validation
# ============================================================

direction_rule = DirectionDecisionRule(
    min_displacement=0.50
)

validation_results = []

print("=" * 110)
print(f"INITIAL DIRECTION RULE — PILOT VALIDATION [{MODEL_NAME}]")
print("=" * 110)

for result in direction_results:

    evidence = {
        "dx": result["dx"],
        "dz": result["dz"],
        "dominant_axis": result["dominant_axis"],
        "dominance_ratio": result["dominance_ratio"],
    }

    auto_result = direction_rule.evaluate(
        evidence=evidence,
        required_direction=result["required_direction"],
    )

    human_label = result["human_label"]
    prediction = auto_result["prediction"]

    if human_label == "UNCERTAIN":
        match = None
        status = "EXCLUDED"
    else:
        match = prediction == human_label
        status = "MATCH" if match else "MISMATCH"

    validation_results.append({
        "model": MODEL_NAME,
        "prompt_id": result["prompt_id"],
        "requirement_index": result["requirement_index"],
        "required_direction": result["required_direction"],
        "human_label": human_label,
        "prediction": prediction,
        "dx": result["dx"],
        "dz": result["dz"],
        "required_displacement": auto_result["required_displacement"],
        "correct_sign": auto_result["correct_sign"],
        "sufficient_displacement": auto_result["sufficient_displacement"],
        "match": match,
        "status": status,
    })

usable = [r for r in validation_results if r["match"] is not None]
correct = sum(1 for r in usable if r["match"])
accuracy = correct / len(usable) if usable else 0.0

false_pass = sum(
    1 for r in usable
    if r["human_label"] == "FAIL" and r["prediction"] == "PASS"
)

false_fail = sum(
    1 for r in usable
    if r["human_label"] == "PASS" and r["prediction"] == "FAIL"
)

print(f"Usable Cases       : {len(usable)}")
print(f"Correct Predictions: {correct}")
print(f"Accuracy           : {accuracy:.3f}")
print(f"False PASS         : {false_pass}")
print(f"False FAIL         : {false_fail}")
print(f"UNCERTAIN Excluded : {len(validation_results) - len(usable)}")

mismatches = [r for r in usable if not r["match"]]

if mismatches:
    print("\nMISMATCH CASES")
    print("-" * 110)
    for r in mismatches:
        print(
            r["prompt_id"],
            "| Required:", r["required_direction"],
            "| Human:", r["human_label"],
            "| Auto:", r["prediction"],
            "| Required displacement:",
            f"{r['required_displacement']:.4f} m",
        )
else:
    print("\nNo PASS/FAIL mismatches found for this model.")

## STEP 10F — Cross-model Direction Validation

Now apply the same notebook and the same Initial Direction Rule to the other models.

For each model, the team member should:

1. Change `MODEL_NAME` in STEP 4 to their model name.
2. Prepare `.npy` files in the common Standardised Motion format.
3. Create or load Human Gold Labels for that model's Pilot motions.
4. Run STEP 10A–10E using the **same Direction Rule and the same 0.50 m candidate threshold**.
5. Record all mismatch cases and their evidence.

### Purpose of Cross-model Validation

The goal is to determine whether the Initial Rule developed from the first Pilot model also agrees with Human Gold on other models.

If mismatches occur, inspect the evidence before changing the rule. Determine whether:

- the threshold is inappropriate,
- an additional condition is required, or
- a complex motion needs event/segment-level evaluation.

**Do not change the threshold simply because one mismatch appears.**

After results from the Pilot models have been compared, refine the rule only if supported by the cross-model evidence. If the rule is changed, re-test the updated rule across the Pilot models.

Once the evaluator design is finalised, **freeze the Final Direction Rule**.

After Rule Freeze, apply exactly the same rule to the remaining Benchmark Prompts without further tuning.

### Important Design Principle

Human Gold Labels are created separately for each model, because each model generates different motions.

However, the final Automatic Evaluator Rule must be **shared across all models**.

This ensures that different T2M models are compared using the same evaluation criteria.

## STEP 11 — Body Side Requirements (`eval_body_side.py`, owner: member B)

`BodySideEvaluator` decides **which side of the body — left / right / both — performs the action** (C2-09/10 kick, C2-19/20 reach, C4-19 raise right hand, C5-16 both hands).

- Left / right come from the anatomical joint labels (joint 20 = left wrist), **not** from the world X axis → independent of facing, walking and turning.
- Evidence: limb activity = how far the wrist / ankle leaves its neutral hanging position (relative to shoulder / hip, divided by limb length; 95th percentile over the clip). ≈0.5 walking arm swing, ≈1.1 side kick, ≈2.0 arm overhead.
- Laterality index `LI = (A_left − A_right) / (A_left + A_right)`.
- Thresholds (`min_activity`, `side_margin`, `both_max_imbalance`) are calibrated against Human Gold in 11B, like the Direction workflow.

Full method description: docstring of `evaluation/eval_body_side.py`.

### STEP 11A — Body Side Evidence vs Human Gold

In [ ]:
body_side_results = eval_body_side.collect_body_side_evidence(evaluation_cases, HUMAN_GOLD_LABELS)

print("=" * 100)
print(f"BODY SIDE EVIDENCE — {MODEL_NAME}")
print("=" * 100)
header = f"{'Prompt':<7} {'Req':<6} {'Limb':<5} {'A_left':>7} {'A_right':>8} {'LI':>7} {'Raw':<12} {'Human':<10} Source"
print(header)
print("-" * len(header))
for r in body_side_results:
    print(f"{r['prompt_id']:<7} {r['required_side']:<6} {r['limb']:<5} "
          f"{r['left_activity']:>7.2f} {r['right_activity']:>8.2f} {r['laterality_index']:>+7.2f} "
          f"{r['predicted_side_raw']:<12} {str(r['human_label']):<10} {r['limb_source']}")
labels = [r["human_label"] for r in body_side_results]
print("\nHuman labels — PASS:", labels.count("PASS"), "| FAIL:", labels.count("FAIL"),
      "| UNCERTAIN:", labels.count("UNCERTAIN"), "| missing:", labels.count(None))

### STEP 11B — Body Side Threshold Calibration

Grid search against Human Gold (`UNCERTAIN` excluded). Best = highest accuracy → fewest false PASS → closest to the provisional values. With fewer than 10 labelled cases the result is marked `calibrated_small_sample`: recalibrate when more models are labelled, then freeze.

In [ ]:
calibration = eval_body_side.calibrate_body_side_thresholds(body_side_results)
BODY_SIDE_THRESHOLDS = calibration["thresholds"]
BODY_SIDE_THRESHOLD_STATUS = calibration["status"]

print("=" * 100)
print("BODY SIDE THRESHOLD CALIBRATION")
print("=" * 100)
print("Usable Human-labelled cases:", calibration["usable"])
if calibration["usable"]:
    print(f"Best accuracy: {calibration['correct']}/{calibration['usable']} "
          f"| false PASS: {calibration['false_pass']} "
          f"| combinations reaching it: {calibration['n_best']} of {calibration['grid_size']}")
    for r in calibration["cases"]:
        mark = "✅" if r["predicted"] == r["human_label"] else "❌"
        print(f"  {r['prompt_id']:<7} required={r['required_side']:<5} human={r['human_label']:<4} "
              f"predicted={r['predicted']:<4} {mark}")
else:
    print("No PASS / FAIL labels — provisional thresholds kept.")
print("\nSelected thresholds:", BODY_SIDE_THRESHOLDS)
print("Threshold status   :", BODY_SIDE_THRESHOLD_STATUS)

### STEP 11C — Body Side Evaluation

In [ ]:
body_side_final_results = eval_body_side.evaluate_body_side(
    evaluation_cases, BODY_SIDE_THRESHOLDS, BODY_SIDE_THRESHOLD_STATUS, HUMAN_GOLD_LABELS)

print("=" * 100)
print(f"BODY SIDE EVALUATION — {MODEL_NAME}")
print("=" * 100)
for r in body_side_final_results:
    agree = ""
    if r["human_label"] in {"PASS", "FAIL"}:
        agree = "✅" if r["pass_fail"] == r["human_label"] else "❌"
    print(f"{r['prompt_id']:<7} body_side={r['expected_value']:<5} -> {r['pass_fail']:<4} "
          f"(predicted {r['predicted_side']}, human {r['human_label']}) {agree}")
    print(f"        {r['reason']}")

labelled = [r for r in body_side_final_results if r["human_label"] in {"PASS", "FAIL"}]
print(f"\nPASS: {sum(r['pass_fail'] == 'PASS' for r in body_side_final_results)}/{len(body_side_final_results)}")
if labelled:
    print(f"Agreement with Human Gold: {sum(r['pass_fail'] == r['human_label'] for r in labelled)}/{len(labelled)}")

import json
out_path = Path(f"/content/{MODEL_NAME}_body_side_results.json")
out_path.write_text(json.dumps({"framework_commit": commit, "results": body_side_final_results},
                               indent=2, ensure_ascii=False, default=float))
print("Saved:", out_path)

## Adding another evaluator

1. Copy `evaluation/eval_template.py` to `evaluation/eval_<name>.py` on your own branch and implement it.
2. Add `import eval_<name>` (and the reload) to STEP 0.
3. Add a `STEP 1x` section here that calls your evaluator — keep the logic in your `.py` file, only printing here.
4. Add `tests/test_<name>.py`, open a Pull Request, get a review, merge.